# 07 — Capa Oro: Agregaciones y KPIs

**Proyecto:** Mexico Economic Pipeline  
**Propósito:** Leer el panel analítico Plata, agregar a nivel año/trimestre
y producir tablas Gold listas para análisis y visualización.

**Pipeline:** `ingesta_general.ipynb` → `05_limpieza_enoe.ipynb` → `06_integracion.ipynb` → **`07_capa_oro.ipynb`**

**Prerequisito:** haber ejecutado `06_integracion.ipynb` completo.

**Tablas Gold generadas:**
```
data/gold/
├── salario_real/          ← salario real por año/trimestre/sexo/escolaridad/edad
├── empleo_sector/         ← ocupados y salario por sector económico
├── macro_trimestral/      ← IGAE e INPC agregados, inflación calculada
├── brecha_salarial/       ← brecha H vs M por grupo de edad
└── kpi_anual/             ← tabla maestra: un KPI por año, todas las fuentes
```

## 0 · Imports y configuración de Spark

In [ ]:
import os
from pathlib import Path

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType

spark = (
    SparkSession.builder
    .appName('07_capa_oro')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark', spark.version)

In [ ]:
# ── Rutas ───────────────────────────────────────────────────────────────────
BASE   = Path(os.getcwd()).parent
SILVER = BASE / 'data' / 'silver'
GOLD   = BASE / 'data' / 'gold'

PANEL_IN = str(SILVER / 'panel_enoe_inpc')   # salida de 06_integracion

GOLD_SALARIO  = str(GOLD / 'salario_real')
GOLD_EMPLEO   = str(GOLD / 'empleo_sector')
GOLD_MACRO    = str(GOLD / 'macro_trimestral')
GOLD_BRECHA   = str(GOLD / 'brecha_salarial')
GOLD_KPI      = str(GOLD / 'kpi_anual')

for d in [GOLD_SALARIO, GOLD_EMPLEO, GOLD_MACRO, GOLD_BRECHA, GOLD_KPI]:
    Path(d).mkdir(parents=True, exist_ok=True)

print('Panel entrada:', PANEL_IN)
print('Gold salida  :', str(GOLD))

---
## 1 · Extracción — leer panel Plata

In [ ]:
# ── 1.1 Cargar panel analítico ───────────────────────────────────────────────
panel = spark.read.parquet(PANEL_IN)

filas_panel = panel.count()
print(f'Filas    : {filas_panel:,}')
print(f'Columnas : {len(panel.columns)}')
print(f'Periodo  :', panel.agg(F.min('anio'), F.max('anio')).collect()[0][:])

In [ ]:
# ── 1.2 Verificar columnas clave disponibles ─────────────────────────────────
COLS_ESPERADAS = [
    'anio', 'trimestre', 'sex', 'eda', 'cs_p13_1',
    'clase1', 'ingocup', 'salario_real', 'inpc_trim', 'igae_total'
]
presentes  = [c for c in COLS_ESPERADAS if c in panel.columns]
ausentes   = [c for c in COLS_ESPERADAS if c not in panel.columns]

print(f'Columnas clave presentes : {presentes}')
if ausentes:
    print(f'ADVERTENCIA — ausentes   : {ausentes}')
else:
    print('OK — todas las columnas clave disponibles')

In [ ]:
# ── 1.3 Crear columnas de agrupación ─────────────────────────────────────────
# Se añaden etiquetas legibles que se reutilizan en todas las tablas Gold.

panel = (
    panel
    .withColumn(
        'sexo',
        F.when(F.col('sex') == 1, 'Hombre').otherwise('Mujer')
    )
    .withColumn(
        'nivel_escolar',
        F.when(F.col('cs_p13_1').cast(IntegerType()) <= 2, 'Sin instr./Primaria')
         .when(F.col('cs_p13_1').cast(IntegerType()) <= 4, 'Secundaria/Prepa')
         .when(F.col('cs_p13_1').cast(IntegerType()) <= 6, 'Técnico/Licenciatura')
         .otherwise('Posgrado')
    )
    .withColumn(
        'grupo_edad',
        F.when(F.col('eda').between(15, 24), '15-24')
         .when(F.col('eda').between(25, 34), '25-34')
         .when(F.col('eda').between(35, 44), '35-44')
         .when(F.col('eda').between(45, 54), '45-54')
         .otherwise('55+')
    )
)

print('Columnas de agrupación creadas OK')
panel.select('sexo', 'nivel_escolar', 'grupo_edad').show(5)

---
## 2 · Gold 1 — Salario real por año / trimestre / sexo / escolaridad / edad

Solo se incluyen ocupados (`clase1 = 1`) con ingreso positivo deflactado.

**Fórmula aplicada en Plata:**  
`salario_real = ingocup × (inpc_base / inpc_trim)` — base T4-2018

In [ ]:
# ── 2.1 Agregar salario real ─────────────────────────────────────────────────
gold_salario = (
    panel
    .filter(
        (F.col('clase1') == 1) &
        F.col('salario_real').isNotNull() &
        (F.col('salario_real') > 0)
    )
    .groupBy('anio', 'trimestre', 'sexo', 'nivel_escolar', 'grupo_edad')
    .agg(
        F.round(F.avg('salario_real'),  2).alias('salario_real_prom'),
        F.round(F.avg('ingocup'),       2).alias('ingreso_nominal_prom'),
        F.percentile_approx('salario_real', 0.5).alias('salario_real_mediana'),
        F.count('*').alias('n_ocupados')
    )
    .orderBy('anio', 'trimestre', 'sexo', 'nivel_escolar', 'grupo_edad')
)

print(f'Gold salario — filas: {gold_salario.count():,}')
gold_salario.show(10, truncate=False)

In [ ]:
# ── 2.2 Escribir Parquet Gold ────────────────────────────────────────────────
(
    gold_salario
    .write
    .mode('overwrite')
    .partitionBy('anio')
    .parquet(GOLD_SALARIO)
)

size_mb = sum(f.stat().st_size for f in Path(GOLD_SALARIO).rglob('*.parquet')) / 1_048_576
print(f'Escrito en : {GOLD_SALARIO}')
print(f'Tamaño     : {size_mb:.1f} MB')

---
## 3 · Gold 2 — Empleo y ocupación por sector

Distribución de ocupados y salario promedio por sector económico (campo `rama`),
año, trimestre y sexo.

In [ ]:
# ── 3.1 Mapeo de rama ENOE → sector legible ──────────────────────────────────
# rama: 1=Agropecuario, 2=Industria extractiva, 3=Industria manufacturera,
#        4=Elect/Gas/Agua, 5=Construcción, 6=Comercio, 7=Restaurantes/hoteles,
#        8=Transporte, 9=Serv. financieros, 10=Serv. sociales, 11=Otros servicios

sector_expr = (
    F.when(F.col('rama').cast(IntegerType()) == 1,  'Agropecuario')
     .when(F.col('rama').cast(IntegerType()) == 2,  'Ind. extractiva')
     .when(F.col('rama').cast(IntegerType()) == 3,  'Manufactura')
     .when(F.col('rama').cast(IntegerType()) == 4,  'Electricidad/Agua')
     .when(F.col('rama').cast(IntegerType()) == 5,  'Construcción')
     .when(F.col('rama').cast(IntegerType()) == 6,  'Comercio')
     .when(F.col('rama').cast(IntegerType()) == 7,  'Rest./Hoteles')
     .when(F.col('rama').cast(IntegerType()) == 8,  'Transporte')
     .when(F.col('rama').cast(IntegerType()) == 9,  'Serv. financieros')
     .when(F.col('rama').cast(IntegerType()) == 10, 'Serv. sociales')
     .when(F.col('rama').cast(IntegerType()) == 11, 'Otros servicios')
     .otherwise('No especificado')
)

gold_empleo = (
    panel
    .filter(F.col('clase1') == 1)   # solo ocupados
    .withColumn('sector', sector_expr)
    .groupBy('anio', 'trimestre', 'sexo', 'sector')
    .agg(
        F.count('*').alias('n_ocupados'),
        F.round(F.avg('salario_real'), 2).alias('salario_real_prom'),
        F.round(F.avg('hrsocup'),      2).alias('horas_trab_prom')
    )
    .orderBy('anio', 'trimestre', 'sector', 'sexo')
)

print(f'Gold empleo — filas: {gold_empleo.count():,}')
gold_empleo.show(10, truncate=False)

In [ ]:
# ── 3.2 Escribir Parquet Gold ────────────────────────────────────────────────
(
    gold_empleo
    .write
    .mode('overwrite')
    .partitionBy('anio')
    .parquet(GOLD_EMPLEO)
)

size_mb = sum(f.stat().st_size for f in Path(GOLD_EMPLEO).rglob('*.parquet')) / 1_048_576
print(f'Escrito en : {GOLD_EMPLEO}')
print(f'Tamaño     : {size_mb:.1f} MB')

---
## 4 · Gold 3 — Indicadores macroeconómicos trimestrales (IGAE + INPC)

Agrega IGAE e INPC a nivel trimestre y calcula inflación interanual.

In [ ]:
# ── 4.1 Agregar IGAE e INPC por trimestre ───────────────────────────────────
w_trim = Window.orderBy('anio', 'trimestre')

gold_macro = (
    panel
    .groupBy('anio', 'trimestre')
    .agg(
        F.round(F.avg('inpc_trim'),   4).alias('inpc_prom'),
        F.round(F.avg('igae_total'),  4).alias('igae_prom'),
        F.count('*').alias('n_obs')
    )
    .orderBy('anio', 'trimestre')
)

# Inflación interanual: variación del INPC respecto al mismo trimestre del año anterior
gold_macro = gold_macro.withColumn(
    'inflacion_interanual_pct',
    F.round(
        (F.col('inpc_prom') - F.lag('inpc_prom', 4).over(w_trim))
        / F.lag('inpc_prom', 4).over(w_trim) * 100,
        2
    )
).withColumn(
    'crecimiento_igae_pct',
    F.round(
        (F.col('igae_prom') - F.lag('igae_prom', 4).over(w_trim))
        / F.lag('igae_prom', 4).over(w_trim) * 100,
        2
    )
)

print(f'Gold macro — filas: {gold_macro.count():,}')
gold_macro.show(30, truncate=False)

In [ ]:
# ── 4.2 Escribir Parquet Gold ────────────────────────────────────────────────
(
    gold_macro
    .write
    .mode('overwrite')
    .parquet(GOLD_MACRO)   # sin partición — son ~85 filas
)

size_mb = sum(f.stat().st_size for f in Path(GOLD_MACRO).rglob('*.parquet')) / 1_048_576
print(f'Escrito en : {GOLD_MACRO}')
print(f'Tamaño     : {size_mb:.2f} MB')

---
## 5 · Gold 4 — Brecha salarial de género por grupo de edad

Pivot Hombre vs Mujer para calcular el diferencial porcentual
del salario real por año, trimestre y grupo de edad.

In [ ]:
# ── 5.1 Pivot sexo → columnas Hombre / Mujer ────────────────────────────────
gold_brecha = (
    panel
    .filter(
        (F.col('clase1') == 1) &
        F.col('salario_real').isNotNull() &
        (F.col('salario_real') > 0)
    )
    .groupBy('anio', 'trimestre', 'grupo_edad')
    .pivot('sexo', ['Hombre', 'Mujer'])
    .agg(F.round(F.avg('salario_real'), 2))
    .withColumnRenamed('Hombre', 'sal_real_hombre')
    .withColumnRenamed('Mujer',  'sal_real_mujer')
    .withColumn(
        'brecha_pct',
        F.round(
            (F.col('sal_real_hombre') - F.col('sal_real_mujer'))
            / F.col('sal_real_hombre') * 100,
            2
        )
    )
    .orderBy('anio', 'trimestre', 'grupo_edad')
)

print(f'Gold brecha — filas: {gold_brecha.count():,}')
gold_brecha.show(15, truncate=False)

In [ ]:
# ── 5.2 Escribir Parquet Gold ────────────────────────────────────────────────
(
    gold_brecha
    .write
    .mode('overwrite')
    .partitionBy('anio')
    .parquet(GOLD_BRECHA)
)

size_mb = sum(f.stat().st_size for f in Path(GOLD_BRECHA).rglob('*.parquet')) / 1_048_576
print(f'Escrito en : {GOLD_BRECHA}')
print(f'Tamaño     : {size_mb:.2f} MB')

---
## 6 · Gold 5 — KPI Maestro Anual

Una fila por año con todos los indicadores clave cruzados.
Esta es la tabla principal para dashboards y presentaciones.

In [ ]:
# ── 6.1 Salario real anual por sexo (pivot) ──────────────────────────────────
sal_sexo_anual = (
    panel
    .filter(
        (F.col('clase1') == 1) &
        F.col('salario_real').isNotNull() &
        (F.col('salario_real') > 0)
    )
    .groupBy('anio')
    .pivot('sexo', ['Hombre', 'Mujer'])
    .agg(F.round(F.avg('salario_real'), 2))
    .withColumnRenamed('Hombre', 'sal_real_hombre')
    .withColumnRenamed('Mujer',  'sal_real_mujer')
)

# ── 6.2 Indicadores macroeconómicos anuales ──────────────────────────────────
macro_anual = (
    panel
    .groupBy('anio')
    .agg(
        F.round(F.avg('inpc_trim'),   2).alias('inpc_anual'),
        F.round(F.avg('igae_total'),  2).alias('igae_anual'),
        F.round(
            F.avg(F.when(
                (F.col('clase1') == 1) & (F.col('salario_real') > 0),
                F.col('salario_real')
            )), 2
        ).alias('salario_real_total'),
        F.count('*').alias('n_personas_total'),
        F.sum(F.when(F.col('clase1') == 1, 1).otherwise(0)).alias('n_ocupados'),
        F.sum(F.when(F.col('clase1') == 2, 1).otherwise(0)).alias('n_desocupados')
    )
    .withColumn(
        'tasa_desocupacion_pct',
        F.round(
            F.col('n_desocupados') / (F.col('n_ocupados') + F.col('n_desocupados')) * 100,
            2
        )
    )
)

print('Macro anual OK')
macro_anual.orderBy('anio').show(25, truncate=False)

In [ ]:
# ── 6.3 Unir y calcular variaciones interanuales ─────────────────────────────
w_anio = Window.orderBy('anio')

gold_kpi = (
    macro_anual
    .join(sal_sexo_anual, on='anio', how='left')
    .withColumn(
        'brecha_salarial_pct',
        F.round(
            (F.col('sal_real_hombre') - F.col('sal_real_mujer'))
            / F.col('sal_real_hombre') * 100,
            2
        )
    )
    .withColumn(
        'var_salario_real_pct',
        F.round(
            (F.col('salario_real_total') - F.lag('salario_real_total', 1).over(w_anio))
            / F.lag('salario_real_total', 1).over(w_anio) * 100,
            2
        )
    )
    .withColumn(
        'inflacion_anual_pct',
        F.round(
            (F.col('inpc_anual') - F.lag('inpc_anual', 1).over(w_anio))
            / F.lag('inpc_anual', 1).over(w_anio) * 100,
            2
        )
    )
    .withColumn(
        'crecimiento_igae_pct',
        F.round(
            (F.col('igae_anual') - F.lag('igae_anual', 1).over(w_anio))
            / F.lag('igae_anual', 1).over(w_anio) * 100,
            2
        )
    )
    .orderBy('anio')
)

print(f'Gold KPI anual — filas: {gold_kpi.count():,}')
gold_kpi.show(25, truncate=False)

In [ ]:
# ── 6.4 Escribir Parquet Gold ────────────────────────────────────────────────
(
    gold_kpi
    .write
    .mode('overwrite')
    .parquet(GOLD_KPI)   # sin partición — son ~22 filas
)

size_mb = sum(f.stat().st_size for f in Path(GOLD_KPI).rglob('*.parquet')) / 1_048_576
print(f'Escrito en : {GOLD_KPI}')
print(f'Tamaño     : {size_mb:.3f} MB')

---
## 7 · Validación round-trip y resumen

In [ ]:
# ── 7.1 Verificar que todas las tablas Gold son legibles ─────────────────────
tablas = [
    ('salario_real',   GOLD_SALARIO),
    ('empleo_sector',  GOLD_EMPLEO),
    ('macro_trimestral', GOLD_MACRO),
    ('brecha_salarial', GOLD_BRECHA),
    ('kpi_anual',      GOLD_KPI),
]

print('Validación round-trip:')
for nombre, ruta in tablas:
    df = spark.read.parquet(ruta)
    print(f'  {nombre:<22} filas={df.count():>8,}  cols={len(df.columns)}')

In [ ]:
# ── 7.2 Resumen final ────────────────────────────────────────────────────────
print('=' * 60)
print('RESUMEN — CAPA ORO')
print('=' * 60)
print()
print('ENTRADA:')
print(f'  Panel Plata : {filas_panel:,} filas — data/silver/panel_enoe_inpc/')
print()
print('TABLAS GOLD GENERADAS:')
for nombre, ruta in tablas:
    df  = spark.read.parquet(ruta)
    mb  = sum(f.stat().st_size for f in Path(ruta).rglob('*.parquet')) / 1_048_576
    print(f'  {nombre:<22} {df.count():>8,} filas  {mb:>6.2f} MB  {ruta}')
print()
print('DECISIONES DE AGREGACIÓN:')
print('  1. Salario real — solo ocupados (clase1=1) con ingreso > 0')
print('  2. Sector económico — mapeado desde campo rama (11 categorías)')
print('  3. IGAE — base 2018=100 (coincide con base de deflactación INPC)')
print('  4. Inflación — variación interanual del INPC trimestral')
print('  5. Brecha salarial — pivot H vs M, cálculo % sobre salario masculino')
print('  6. KPI Maestro — tasa de desocupación y variaciones YoY incluidas')
print('=' * 60)
print('Siguiente paso: análisis y visualización.')
spark.stop()